<a href="https://colab.research.google.com/github/FelixVVu/jinke/blob/main/jinke_colab_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Jinke production data generation

This notebook reuses the completed 50-minute cache and generates only the missing 10/20/30/40-minute ORS caches. It never calls ORS unless you explicitly change `RUN_ORS = False` to `True` in Stage 3 or Stage 4.

Run the stages in order. Each stage is a separately runnable cell. Do not use **Run all**, because the two live stages require a deliberate opt-in.

## Stage 1 — setup

**Change:** nothing.

**Run:** the single code cell immediately below, labelled `STAGE 1 — SETUP`.

This cell clones or fast-forward-updates `FelixVVu/jinke`, installs `requirements.txt`, mounts Google Drive, loads `ORS_API_KEY` from Colab Secrets, and creates `MyDrive/Jinke50min/ors_cache_multilimit_v1/` automatically. It does **not** call ORS.

In [ ]:
# STAGE 1 — SETUP (change nothing; run this entire cell)
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/FelixVVu/jinke.git"
REPO_REF = "main"
REPO_DIR = Path("/content/jinke")

if (REPO_DIR / ".git").is_dir():
    dirty = subprocess.run(
        ["git", "-C", str(REPO_DIR), "status", "--porcelain"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    if dirty:
        raise RuntimeError(
            f"{REPO_DIR} has local changes. Start a fresh Colab runtime "
            "or remove those changes before rerunning setup."
        )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", REPO_REF],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", REPO_REF],
        check=True,
    )
elif REPO_DIR.exists():
    raise RuntimeError(
        f"{REPO_DIR} exists but is not a git checkout. "
        "Start a fresh Colab runtime and rerun this cell."
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
        check=True,
    )

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(REPO_DIR / "requirements.txt"),
    ],
    check=True,
)
os.chdir(REPO_DIR)

from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    ORS_API_KEY = userdata.get("ORS_API_KEY")
except Exception as exc:
    ORS_API_KEY = None
    print(
        "ORS_API_KEY is not currently available from Colab Secrets. "
        "Dry run still works; Stage 3 and Stage 4 will stop until the "
        "secret exists and Notebook access is enabled."
    )

BASE_DIR = Path("/content/drive/MyDrive/Jinke50min")
LEGACY_50_CACHE_DIR = BASE_DIR / "ors_cache_50min"
MULTILIMIT_CACHE_DIR = BASE_DIR / "ors_cache_multilimit_v1"
AUDIT_OUTPUT_DIR = BASE_DIR / "audit_outputs"
WEB_DATA_DIR = REPO_DIR / "web" / "public" / "data"

# The user never needs to create these folders manually.
MULTILIMIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

from dataclasses import replace
from pprint import pprint
from pipeline.generate import (
    Config,
    assert_all_caches_complete,
    build_outputs,
    cache_status,
    fill_cache,
    load_stations,
)

RUN_ORS = False
MAX_ORS_CALLS = 200
ORS_REQUEST_INTERVAL = 3.5

BASE_CONFIG = Config(
    legacy_cache_dir=LEGACY_50_CACHE_DIR,
    cache_dir=MULTILIMIT_CACHE_DIR,
    web_data_dir=WEB_DATA_DIR,
    audit_dir=AUDIT_OUTPUT_DIR,
    dry_run=True,
    max_calls=MAX_ORS_CALLS,
    request_interval=ORS_REQUEST_INTERVAL,
    min_legacy_50_accepted=150,
    test_mode=False,
)
rows = load_stations(BASE_CONFIG)

print("Setup complete.")
print("Repository:", REPO_DIR)
print("Production Sheet rows:", len(rows))
print("Legacy 50-minute cache:", LEGACY_50_CACHE_DIR)
print("New 10/20/30/40 cache:", MULTILIMIT_CACHE_DIR)
print("Audit ZIP:", AUDIT_OUTPUT_DIR / "web-data.zip")
print("ORS secret loaded:", bool(ORS_API_KEY))
print("Defaults: RUN_ORS=False, MAX_ORS_CALLS=200, interval=3.5 seconds")

## Stage 2 — dry run

**Change:** nothing.

**Run:** the single code cell immediately below, labelled `STAGE 2 — DRY RUN`.

This cell makes zero ORS requests. It validates the readable legacy files such as `金科路_3000s.json` and estimates the missing lower-limit calls. Expected values are approximately:

- accepted legacy 50-minute files: **175**
- new 10/20/30/40-minute requests: **173**

If the accepted legacy count is close to zero—or if any required legacy file is missing or invalid—the cell stops with a clear error before any ORS request can occur.

In [ ]:
# STAGE 2 — DRY RUN (change nothing; run this entire cell)
RUN_ORS = False
MAX_ORS_CALLS = 200

dry_cfg = replace(
    BASE_CONFIG,
    dry_run=not RUN_ORS,
    max_calls=MAX_ORS_CALLS,
)
dry_report = fill_cache(rows, dry_cfg, api_key=None)

accepted_legacy = dry_report["initial_status"][
    "legacy_50_cache_files_accepted"
]
new_requests_needed = dry_report["initial_missing_requests"]

print("Dry run complete — ORS calls made:", dry_report["ors_calls_made"])
print("Legacy 50-minute caches accepted:", accepted_legacy, "(expected ≈175)")
print("New lower-limit requests needed:", new_requests_needed, "(expected ≈173)")
print("Requests by limit:")
pprint(
    dry_report["initial_status"]["required_cache_files_by_limit"]
)
print("First missing lower-limit requests:")
pprint(dry_report["request_preview"])

## Stage 3 — five-call smoke test

**Change:** in the next cell, change only `RUN_ORS = False` to `RUN_ORS = True`.

**Do not change:** `MAX_ORS_CALLS = 5` or `ORS_REQUEST_INTERVAL = 3.5`.

**Run:** the single code cell immediately below, labelled `STAGE 3 — FIVE-CALL SMOKE TEST`.

Run this only after Stage 2 reports the expected legacy cache. The five-call budget counts actual HTTP attempts, including retries. New files are written only to `ors_cache_multilimit_v1/`; legacy files remain untouched.

In [ ]:
# STAGE 3 — FIVE-CALL SMOKE TEST
# CHANGE ONLY the next line to True when you are ready.
RUN_ORS = False
MAX_ORS_CALLS = 5
ORS_REQUEST_INTERVAL = 3.5

if not RUN_ORS:
    print(
        "Skipped safely. Change only RUN_ORS = False to RUN_ORS = True "
        "in this Stage 3 cell, then rerun this cell."
    )
else:
    if not ORS_API_KEY:
        raise RuntimeError(
            "ORS_API_KEY is missing. Add it in Colab Secrets, enable "
            "Notebook access, then rerun Stage 1 before this cell."
        )
    smoke_cfg = replace(
        BASE_CONFIG,
        dry_run=False,
        max_calls=MAX_ORS_CALLS,
        request_interval=ORS_REQUEST_INTERVAL,
    )
    smoke_report = fill_cache(rows, smoke_cfg, api_key=ORS_API_KEY)
    print("Five-call smoke test finished.")
    print("Actual ORS HTTP calls:", smoke_report["ors_calls_made"])
    print("New cache files written:", smoke_report["new_cache_files_written"])
    print("Remaining lower-limit requests:", smoke_report["remaining_requests"])
    print("Failures:")
    pprint(smoke_report["failures"])

## Stage 4 — full lower-limit run

**Change:** in the next cell, change only `RUN_ORS = False` to `RUN_ORS = True`.

**Do not change:** `MAX_ORS_CALLS = 200` or `ORS_REQUEST_INTERVAL = 3.5`.

**Run:** the single code cell immediately below, labelled `STAGE 4 — FULL LOWER-LIMIT RUN`.

This stage requests only missing 10/20/30/40-minute caches. It never regenerates the 50-minute legacy files. If `Remaining lower-limit requests` is not zero because of retries, quota, or a runtime interruption, rerun this same cell; valid completed files are reused.

In [ ]:
# STAGE 4 — FULL LOWER-LIMIT RUN
# CHANGE ONLY the next line to True when you are ready.
RUN_ORS = False
MAX_ORS_CALLS = 200
ORS_REQUEST_INTERVAL = 3.5

if not RUN_ORS:
    print(
        "Skipped safely. Change only RUN_ORS = False to RUN_ORS = True "
        "in this Stage 4 cell, then rerun this cell."
    )
else:
    if not ORS_API_KEY:
        raise RuntimeError(
            "ORS_API_KEY is missing. Add it in Colab Secrets, enable "
            "Notebook access, then rerun Stage 1 before this cell."
        )
    full_cfg = replace(
        BASE_CONFIG,
        dry_run=False,
        max_calls=MAX_ORS_CALLS,
        request_interval=ORS_REQUEST_INTERVAL,
    )
    full_report = fill_cache(rows, full_cfg, api_key=ORS_API_KEY)
    print("Full lower-limit pass finished.")
    print("Actual ORS HTTP calls:", full_report["ors_calls_made"])
    print("New cache files written:", full_report["new_cache_files_written"])
    print("Remaining lower-limit requests:", full_report["remaining_requests"])
    print("Failures:")
    pprint(full_report["failures"])

    if full_report["remaining_requests"]:
        print(
            "Not complete yet. Keep RUN_ORS=True and rerun this Stage 4 "
            "cell. Existing valid caches will be reused."
        )
    else:
        print("All lower-limit caches are complete. Continue to Stage 5.")

## Stage 5 — validation and export

**Change:** nothing.

**Run:** the single code cell immediately below, labelled `STAGE 5 — VALIDATION AND EXPORT`, only after Stage 4 reports zero remaining requests.

This cell makes zero ORS requests. It validates every required real cache, builds all five layers with Shapely Polygon/MultiPolygon union, and writes:

`MyDrive/Jinke50min/audit_outputs/web-data.zip`

The ZIP is checked to contain exactly `manifest.json`, `reach-areas.geojson`, and `stations.geojson`. `production_data=true` is written only after every check succeeds.

In [ ]:
# STAGE 5 — VALIDATION AND EXPORT (change nothing; run this entire cell)
import zipfile

export_cfg = replace(
    BASE_CONFIG,
    dry_run=True,
    max_calls=MAX_ORS_CALLS,
    request_interval=ORS_REQUEST_INTERVAL,
)

validation_status = assert_all_caches_complete(rows, export_cfg)
manifest = build_outputs(rows, export_cfg)
zip_path = AUDIT_OUTPUT_DIR / "web-data.zip"

expected_zip_files = {
    "manifest.json",
    "reach-areas.geojson",
    "stations.geojson",
}
with zipfile.ZipFile(zip_path) as archive:
    actual_zip_files = set(archive.namelist())

if actual_zip_files != expected_zip_files:
    raise RuntimeError(
        f"ZIP contents are wrong: {sorted(actual_zip_files)}; "
        f"expected {sorted(expected_zip_files)}"
    )
if manifest.get("production_data") is not True:
    raise RuntimeError("Export finished without production_data=true.")
if manifest.get("all_five_layers_complete") is not True:
    raise RuntimeError("Export finished without all five complete layers.")

print("Validation and production export complete.")
print("Legacy 50-minute caches accepted:", validation_status[
    "legacy_50_cache_files_accepted"
])
print("Lower-limit cache hits:", validation_status["modern_cache_hits"])
print("Layers:", manifest["limits"])
print("production_data:", manifest["production_data"])
print("ZIP:", zip_path)
print("ZIP contents:", sorted(actual_zip_files))